## Задание на 17-18 семинар

In [1]:
pip install psycopg2

   ---------------------------------------- 0.0/2.8 MB ? eta -:--:--
   ---------------------------------------- 0.0/2.8 MB ? eta -:--:--
    --------------------------------------- 0.1/2.8 MB 825.8 kB/s eta 0:00:04
   -- ------------------------------------- 0.2/2.8 MB 1.7 MB/s eta 0:00:02
   ---- ----------------------------------- 0.3/2.8 MB 2.1 MB/s eta 0:00:02
   ------- -------------------------------- 0.5/2.8 MB 2.5 MB/s eta 0:00:01
   ------------- -------------------------- 1.0/2.8 MB 3.8 MB/s eta 0:00:01
   -------------- ------------------------- 1.0/2.8 MB 3.7 MB/s eta 0:00:01
   -------------- ------------------------- 1.0/2.8 MB 3.7 MB/s eta 0:00:01
   -------------- ------------------------- 1.0/2.8 MB 2.6 MB/s eta 0:00:01
   ------------------------------ --------- 2.1/2.8 MB 4.8 MB/s eta 0:00:01
   -------------------------------- ------- 2.3/2.8 MB 4.8 MB/s eta 0:00:01
   -------------------------------- ------- 2.3/2.8 MB 4.8 MB/s eta 0:00:01
   ---------------------

In [3]:
import psycopg2
from psycopg2 import sql

#### Подключения к БД

In [12]:
DB_CONFIG = {
    'dbname': 'dental_clinic',
    'user': 'postgres',
    'password': 'root',
    'host': '127.0.0.1',
    'port': 5432
}

### 1. Создание таблиц

In [14]:
def create_tables():
    """Создаёт таблицы Patients, Services, Appointments с первичными и внешними ключами."""
    conn = psycopg2.connect(**DB_CONFIG)
    cur = conn.cursor()
    # Удаляем таблицы, если существуют (для чистоты эксперимента)
    cur.execute("""
        DROP TABLE IF EXISTS Appointments CASCADE;
        DROP TABLE IF EXISTS Patients CASCADE;
        DROP TABLE IF EXISTS Services CASCADE;
    """)
    # Таблица Patients
    cur.execute("""
        CREATE TABLE Patients (
            patient_id INTEGER PRIMARY KEY,
            last_name TEXT NOT NULL,
            address TEXT,
            birth_year INTEGER
        );
    """)
    # Таблица Services
    cur.execute("""
        CREATE TABLE Services (
            service_code INTEGER PRIMARY KEY,
            service_name TEXT NOT NULL
        );
    """)
    # Таблица Appointments (оказанные услуги)
    cur.execute("""
        CREATE TABLE Appointments (
            patient_id INTEGER REFERENCES Patients(patient_id) ON DELETE CASCADE,
            service_code INTEGER REFERENCES Services(service_code) ON DELETE CASCADE,
            appointment_time TIME NOT NULL,
            cost NUMERIC(10,2) NOT NULL,
            PRIMARY KEY (patient_id, service_code, appointment_time)
        );
    """)
    conn.commit()
    cur.close()
    conn.close()
    print("Таблицы успешно созданы.")

### 2. Добавление данных

#### 2.1 Patients

In [19]:
def insert_patient_one(patient_id, last_name, address, birth_year):
    """Добавляет одного пациента."""
    conn = psycopg2.connect(**DB_CONFIG)
    cur = conn.cursor()
    cur.execute("""
        INSERT INTO Patients (patient_id, last_name, address, birth_year)
        VALUES (%s, %s, %s, %s)
    """, (patient_id, last_name, address, birth_year))
    conn.commit()
    cur.close()
    conn.close()

def insert_patients_many(patients_list):
    """Добавляет несколько пациентов (список кортежей)."""
    conn = psycopg2.connect(**DB_CONFIG)
    cur = conn.cursor()
    cur.executemany("""
        INSERT INTO Patients (patient_id, last_name, address, birth_year)
        VALUES (%s, %s, %s, %s)
    """, patients_list)
    conn.commit()
    cur.close()
    conn.close()

#### 2.2. Services

In [24]:
def insert_service_one(service_code, service_name):
    conn = psycopg2.connect(**DB_CONFIG)
    cur = conn.cursor()
    cur.execute("""
        INSERT INTO Services (service_code, service_name)
        VALUES (%s, %s)
    """, (service_code, service_name))
    conn.commit()
    cur.close()
    conn.close()

def insert_services_many(services_list):
    conn = psycopg2.connect(**DB_CONFIG)
    cur = conn.cursor()
    cur.executemany("""
        INSERT INTO Services (service_code, service_name)
        VALUES (%s, %s)
    """, services_list)
    conn.commit()
    cur.close()
    conn.close()

#### 2.3. Appointments

In [28]:
def insert_appointment_one(patient_id, service_code, appointment_time, cost):
    conn = psycopg2.connect(**DB_CONFIG)
    cur = conn.cursor()
    cur.execute("""
        INSERT INTO Appointments (patient_id, service_code, appointment_time, cost)
        VALUES (%s, %s, %s, %s)
    """, (patient_id, service_code, appointment_time, cost))
    conn.commit()
    cur.close()
    conn.close()

def insert_appointments_many(appointments_list):
    conn = psycopg2.connect(**DB_CONFIG)
    cur = conn.cursor()
    cur.executemany("""
        INSERT INTO Appointments (patient_id, service_code, appointment_time, cost)
        VALUES (%s, %s, %s, %s)
    """, appointments_list)
    conn.commit()
    cur.close()
    conn.close()

### 3. Выборки

#### 3.1. Выбрать всех пациентов

In [31]:
def select_all_patients():
    conn = psycopg2.connect(**DB_CONFIG)
    cur = conn.cursor()
    cur.execute("SELECT * FROM Patients ORDER BY patient_id")
    rows = cur.fetchall()
    cur.close()
    conn.close()
    return rows

#### 3.2. Выбрать пациентов по id (параметризованный запрос)

In [35]:
def select_patients_by_id(patient_id):
    conn = psycopg2.connect(**DB_CONFIG)
    cur = conn.cursor()
    cur.execute("SELECT * FROM Patients WHERE patient_id = %s", (patient_id,))
    rows = cur.fetchall()
    cur.close()
    conn.close()
    return rows

#### 3.3. Выбрать данные с соединением таблиц (пациент + услуга + стоимость)

In [39]:
def select_patients_with_services(min_cost=None):
    """ Возвращает информацию об оказанных услугах. Если передан min_cost, фильтрует по стоимости >= min_cost."""
    conn = psycopg2.connect(**DB_CONFIG)
    cur = conn.cursor()
    query = """
        SELECT p.patient_id, p.last_name, s.service_name, a.appointment_time, a.cost
        FROM Appointments a
        JOIN Patients p ON a.patient_id = p.patient_id
        JOIN Services s ON a.service_code = s.service_code
    """
    params = []
    if min_cost is not None:
        query += " WHERE a.cost >= %s"
        params.append(min_cost)
    query += " ORDER BY p.patient_id, a.appointment_time"
    cur.execute(query, tuple(params))
    rows = cur.fetchall()
    cur.close()
    conn.close()
    return rows

### 4. Обновление данных

#### Обновить адрес пациента

In [43]:
def update_patient_address(patient_id, new_address):
    conn = psycopg2.connect(**DB_CONFIG)
    cur = conn.cursor()
    cur.execute("UPDATE Patients SET address = %s WHERE patient_id = %s", (new_address, patient_id))
    conn.commit()
    updated = cur.rowcount
    cur.close()
    conn.close()
    return updated

#### Обновить название услуги

In [46]:
def update_service_name(service_code, new_name):
    conn = psycopg2.connect(**DB_CONFIG)
    cur = conn.cursor()
    cur.execute("UPDATE Services SET service_name = %s WHERE service_code = %s", (new_name, service_code))
    conn.commit()
    updated = cur.rowcount
    cur.close()
    conn.close()
    return updated

#### Обновить стоимость приёма (запись идентифицируется по пациенту, услуге, времени)

In [49]:
def update_appointment_cost(patient_id, service_code, appointment_time, new_cost):
    conn = psycopg2.connect(**DB_CONFIG)
    cur = conn.cursor()
    cur.execute("""
        UPDATE Appointments
        SET cost = %s
        WHERE patient_id = %s AND service_code = %s AND appointment_time = %s
    """, (new_cost, patient_id, service_code, appointment_time))
    conn.commit()
    updated = cur.rowcount
    cur.close()
    conn.close()
    return updated

### 5. Удаление данных

#### 5.1. Удаление по условию

In [53]:
def delete_patient_by_id(patient_id):
    """Удаляет пациента и все его записи (каскадно из-за ON DELETE CASCADE)."""
    conn = psycopg2.connect(**DB_CONFIG)
    cur = conn.cursor()
    cur.execute("DELETE FROM Patients WHERE patient_id = %s", (patient_id,))
    conn.commit()
    deleted = cur.rowcount
    cur.close()
    conn.close()
    return deleted

def delete_all_patients():
    """Удаляет всех пациентов (и все связанные записи)."""
    conn = psycopg2.connect(**DB_CONFIG)
    cur = conn.cursor()
    cur.execute("DELETE FROM Patients")
    conn.commit()
    deleted = cur.rowcount
    cur.close()
    conn.close()
    return deleted

#### Аналогичные функции для Services и Appointments (примеры)

In [56]:
def delete_service_by_code(service_code):
    conn = psycopg2.connect(**DB_CONFIG)
    cur = conn.cursor()
    cur.execute("DELETE FROM Services WHERE service_code = %s", (service_code,))
    conn.commit()
    deleted = cur.rowcount
    cur.close()
    conn.close()
    return deleted

def delete_all_services():
    conn = psycopg2.connect(**DB_CONFIG)
    cur = conn.cursor()
    cur.execute("DELETE FROM Services")
    conn.commit()
    deleted = cur.rowcount
    cur.close()
    conn.close()
    return deleted

def delete_appointment(patient_id, service_code, appointment_time):
    conn = psycopg2.connect(**DB_CONFIG)
    cur = conn.cursor()
    cur.execute("""
        DELETE FROM Appointments
        WHERE patient_id = %s AND service_code = %s AND appointment_time = %s
    """, (patient_id, service_code, appointment_time))
    conn.commit()
    deleted = cur.rowcount
    cur.close()
    conn.close()
    return deleted

def delete_all_appointments():
    conn = psycopg2.connect(**DB_CONFIG)
    cur = conn.cursor()
    cur.execute("DELETE FROM Appointments")
    conn.commit()
    deleted = cur.rowcount
    cur.close()
    conn.close()
    return deleted

### 6. Вызовы всех функций и демонстрация

In [59]:
def main():
    # 1. Создаём таблицы
    create_tables()

    # 2. Добавляем данные
    # Пациенты (данные из методички, но в ней есть неточности – приводим корректные)
    patients_data = [
        (1, 'Петров', 'Солнечная, 8, 46', 1989),
        (2, 'Иванов', 'Радищева, 22, 22', 1961),
        (3, 'Потапова', 'Горького, 37, 12', 1968),
        (4, 'Зотов', 'Павлова, 1, 10', 1984),
        (5, 'Ковалева', 'Свободы, 81, 70', 1989),
        (6, 'Сидоров', None, 1989),          # адрес NULL
        (7, 'Фролов', 'Погонава, 65, 6', 1961),
        (8, 'Татаринова', 'Соборная, 2, 10', 1975),
        (9, 'Ильин', 'Урицкого, 67, 3', 1987),
        (10, 'Сафронова', 'Каляева, 13, 20', 1980)
    ]
    insert_patients_many(patients_data)

    # Услуги
    services_data = [
        (100, 'Удаление зубов'),
        (101, 'Лечение зубов'),
        (102, 'Протезирование'),
        (103, 'Отбеливание'),
        (104, 'Чистка полости рта'),
        (105, 'Декоративное украшение зубов'),
        (106, 'Рентгенодиагностика'),
        (107, 'Пародонтология'),
        (108, 'Исправление прикуса'),
        (109, 'Реставрация зубов')
    ]
    insert_services_many(services_data)

    # Оказанные услуги (данные из задания)
    appointments_data = [
        (1, 102, '12:00:00', 600.00),
        (8, 104, '15:00:00', 500.00),
        (9, 109, '10:00:00', 100.00),
        (10, 107, '08:00:00', 250.00),
        (6, 104, '17:00:00', 100.00),
        (2, 105, '21:00:00', 750.00),
        (3, 103, '19:00:00', 400.00),
        (7, 102, '12:00:00', 500.00),
        (4, 106, '10:40:00', 340.00),
        (1, 102, '17:10:00', 560.00),
        (9, 104, '15:00:00', 50.00),
        (10, 107, '08:45:00', 100.00),
        (7, 100, '09:00:00', 250.00),
        (3, 103, '10:30:00', 400.00),
        (2, 103, '11:00:00', 980.00),
        (1, 100, '16:00:00', 120.00),
        (4, 101, '12:40:00', 300.00),
        (9, 100, '14:35:00', 460.00),
        (6, 105, '20:00:00', 900.00)
    ]
    insert_appointments_many(appointments_data)
    print("Данные добавлены.")

    # 3. Выборки
    print("\n--- Все пациенты ---")
    for row in select_all_patients():
        print(row)

    print("\n--- Пациент с id=3 ---")
    for row in select_patients_by_id(3):
        print(row)

    print("\n--- Оказанные услуги со стоимостью >= 500 ---")
    for row in select_patients_with_services(min_cost=500):
        print(row)

    # 4. Обновления
    upd = update_patient_address(5, 'Новый адрес, 1')
    print(f"\nОбновлено адресов: {upd}")
    upd = update_service_name(105, 'Драгоценные камни на зубы')
    print(f"Обновлено названий услуг: {upd}")
    upd = update_appointment_cost(1, 102, '12:00:00', 650.00)
    print(f"Обновлено стоимостей приёмов: {upd}")

    # 5. Удаления (по одному и всех)
    print("\n--- Удаление пациента с id=10 ---")
    deleted = delete_patient_by_id(10)
    print(f"Удалено пациентов: {deleted}")

    print("--- Удаление всех оставшихся пациентов (по сути только id=10 уже удалён, удалит остальных) ---")
    deleted_all = delete_all_patients()
    print(f"Удалено всех пациентов: {deleted_all}")

    # (Завершаем, чтобы не потерять данные – закомментируйте последние удаления, если хотите сохранить БД)
    # При необходимости можно также продемонстрировать удаление услуг/записей.

if __name__ == "__main__":
    main()

Таблицы успешно созданы.
Данные добавлены.

--- Все пациенты ---
(1, 'Петров', 'Солнечная, 8, 46', 1989)
(2, 'Иванов', 'Радищева, 22, 22', 1961)
(3, 'Потапова', 'Горького, 37, 12', 1968)
(4, 'Зотов', 'Павлова, 1, 10', 1984)
(5, 'Ковалева', 'Свободы, 81, 70', 1989)
(6, 'Сидоров', None, 1989)
(7, 'Фролов', 'Погонава, 65, 6', 1961)
(8, 'Татаринова', 'Соборная, 2, 10', 1975)
(9, 'Ильин', 'Урицкого, 67, 3', 1987)
(10, 'Сафронова', 'Каляева, 13, 20', 1980)

--- Пациент с id=3 ---
(3, 'Потапова', 'Горького, 37, 12', 1968)

--- Оказанные услуги со стоимостью >= 500 ---
(1, 'Петров', 'Протезирование', datetime.time(12, 0), Decimal('600.00'))
(1, 'Петров', 'Протезирование', datetime.time(17, 10), Decimal('560.00'))
(2, 'Иванов', 'Отбеливание', datetime.time(11, 0), Decimal('980.00'))
(2, 'Иванов', 'Декоративное украшение зубов', datetime.time(21, 0), Decimal('750.00'))
(6, 'Сидоров', 'Декоративное украшение зубов', datetime.time(20, 0), Decimal('900.00'))
(7, 'Фролов', 'Протезирование', datetime